# E-commerce Olist — EDA + Mineração
Objetivo: segmentar clientes (RFM + KMeans k=4), prever satisfação (árvore + matriz de confusão) e extrair regras de associação (co-ocorrência de categorias) para apoiar decisões de logística e marketing.

## 0. Fonte dos dados
Real: `kaggle datasets download olistbr/brazilian-ecommerce` (9 CSVs). Sem credencial Kaggle neste ambiente, geramos dados sintéticos realistas com o mesmo esquema (seed 42) — entidades orders/order_items/customers/products, nulos de `order_delivered_*` (~3% cancelados) e outliers de frete — documentados abaixo.

import matplotlib.pyplot as plt
import pandas as pd
from src.common import load_csv, save_fig, df_to_html_table
from src.mining import run_kmeans, run_tree, confusion_summary

dict9 = pd.DataFrame([
    ("olist_orders_dataset.csv", "pedidos: status, timestamps de compra/aprovacao/entrega"),
    ("olist_order_items_dataset.csv", "itens: preco, frete, vendedor"),
    ("olist_customers_dataset.csv", "clientes: id, cidade/UF"),
    ("olist_products_dataset.csv", "produtos: categoria, dimensoes"),
    ("olist_order_reviews_dataset.csv", "avaliacoes: review_score 1-5"),
    ("olist_order_payments_dataset.csv", "pagamentos: tipo, parcelas, valor"),
    ("olist_sellers_dataset.csv", "vendedores: cidade/UF"),
    ("olist_geolocation_dataset.csv", "geolocalizacao: CEP/lat-lng"),
    ("product_category_name_translation.csv", "traducao categoria PT-BR -> EN"),
], columns=["arquivo", "conteudo"])
dict9

In [ ]:
import pandas as pd
from src.common import load_csv, save_fig, df_to_html_table
from src.mining import run_kmeans, run_tree, confusion_summary

dict9 = pd.DataFrame([
    ("olist_orders_dataset.csv", "pedidos: status, timestamps de compra/aprovacao/entrega"),
    ("olist_order_items_dataset.csv", "itens: preco, frete, vendedor"),
    ("olist_customers_dataset.csv", "clientes: id, cidade/UF"),
    ("olist_products_dataset.csv", "produtos: categoria, dimensoes"),
    ("olist_order_reviews_dataset.csv", "avaliacoes: review_score 1-5"),
    ("olist_order_payments_dataset.csv", "pagamentos: tipo, parcelas, valor"),
    ("olist_sellers_dataset.csv", "vendedores: cidade/UF"),
    ("olist_geolocation_dataset.csv", "geolocalizacao: CEP/lat-lng"),
    ("product_category_name_translation.csv", "traducao categoria PT-BR -> EN"),
], columns=["arquivo", "conteudo"])
dict9

## 2. Preparação: join + nulos + outliers
Join `orders ⨝ order_items ⨝ customers`; nulos de `order_delivered_customer_date`/`order_delivered_carrier_date` (cancelados) viram flag `is_null_delivered` e saem do cálculo de `delay_days`; outliers de preço/frete (IQR) com clip no p99.

In [ ]:
# dados sinteticos realistas (fallback offline) — ver build em /tmp/opencode/build_olist.py
import numpy as np
rng = np.random.default_rng(42)
# ... geracao de customers/orders/items, join, flags de nulos e clip p99 ...
df = orders.merge(items, on='order_id', how='left')
df['is_null_delivered'] = df['order_delivered_customer_date'].isna()
for col in ['price', 'freight_value']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))
df['freight_ratio'] = df['freight_value'] / df['price']
df['is_late'] = (df['delay_days'] > 0).fillna(False).astype(int)
df.head(10)

## 3. EDA: atraso × review, receita por categoria/UF, frete × distância (6–8 figs `olist_*` via `save_fig`)

In [ ]:
g = df.dropna(subset=['review_score']).groupby('review_score')['delay_days'].mean()
fig, ax = plt.subplots()
ax.bar(g.index.astype(int).astype(str), g.values)
ax.set(xlabel='review_score', ylabel='atraso medio (dias)',
       title='Atraso medio por nota')
save_fig(fig, 'olist_atraso_review.png')

In [ ]:
g = df.groupby('product_category_name')['price'].sum().sort_values().tail(10)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(g.index, g.values)
ax.set(xlabel='receita (R$)', title='Receita por categoria (top 10)')
save_fig(fig, 'olist_receita_categoria.png')

In [ ]:
g = df.groupby('customer_state')['price'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots()
ax.bar(g.index, g.values)
ax.set(xlabel='UF', ylabel='receita (R$)', title='Receita por UF')
save_fig(fig, 'olist_receita_uf.png')

In [ ]:
d = df[['customer_state', 'freight_value']].dropna().sample(1500, random_state=42)
d = d.assign(dist=d['customer_state'].map(DIST_KM))  # distancia aprox. do CD por UF
fig, ax = plt.subplots()
ax.scatter(d['dist'], d['freight_value'], s=6, alpha=0.4)
ax.set(xlabel='distancia aprox. do CD (km, por UF)', ylabel='frete (R$)',
       title='Frete vs distancia aproximada')
save_fig(fig, 'olist_frete_distancia.png')

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].boxplot(items['price']); axes[0].set_title('Preco antes do clip p99')
axes[1].boxplot(items['freight_value']); axes[1].set_title('Frete antes do clip p99')
save_fig(fig, 'olist_outliers_preco_frete.png')

## 4. Mineração: KMeans em RFM (k=4), árvore p/ `review_score>=4` + `confusion_summary`, co-ocorrência de categorias

In [ ]:
from sklearn.preprocessing import StandardScaler
rfm = df.groupby('customer_unique_id').agg(recency=('order_purchase_timestamp', ...),
    frequency=('order_id', 'nunique'), monetary=('price', 'sum'))
X = StandardScaler().fit_transform(rfm[['recency', 'frequency', 'monetary']])
labels, km = run_kmeans(pd.DataFrame(X), k=4, seed=42)
rfm['cluster'] = labels  # KMeans: 4 segmentos de clientes

In [ ]:
from sklearn.model_selection import train_test_split
Xf = feat[['delay_fill', 'freight_value', 'price', 'freight_ratio', 'n_items', 'is_late']]
yf = (feat['review_score'] >= 4).astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.3, random_state=42)
tree = run_tree(Xtr, ytr)
summ = confusion_summary(yte, tree.predict(Xte))
summ  # {'accuracy': ..., 'confusion_matrix': [[...]]}

In [ ]:
# associacao: pares de categorias em pedidos com 2+ itens
cooc = pd.Series(pairs).value_counts().head(10)
cooc

## 5. Decisão (se → então)
1. Se atraso > 7 dias derruba a nota, então reforçar CD no Sudeste.
2. Se o KMeans isola o segmento de alto valor, então fidelizá-lo com frete progressivo.
3. Se razão frete/preço alta prediz nota baixa, então subsidiar frete de longa distância.
4. Se categorias co-ocorrem, então criar bundles com desconto (recomenda-se kits).